# Telugu Text Classifier — Phase 2 training (Colab GPU)

Fine-tunes IndicBERTv2 / MuRIL / XLM-R on the frozen Phase 1 splits.

**Before running:** `Runtime > Change runtime type > T4 GPU`, and set `REPO_URL` below.
Checkpoints are written to Google Drive so they survive runtime disconnects (~9 runs, roughly 1.5–2.5 h total on T4).

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/Telugu-Text-Classifier"  # <-- EDIT
DRIVE_DIR = "/content/drive/MyDrive/telugu-classifier"               # checkpoints + results land here
SEEDS = [42, 7, 123]
MODELS = [
    "ai4bharat/IndicBERTv2-MLM-only",
    "google/muril-base-cased",
    "xlm-roberta-base",
]

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
if not os.path.exists("/content/Telugu-Text-Classifier"):
    !git clone {REPO_URL} /content/Telugu-Text-Classifier
%cd /content/Telugu-Text-Classifier
!git log --oneline | head -5
# Colab already ships CUDA torch; requirements.txt intentionally leaves torch unpinned
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(f"{DRIVE_DIR}/outputs", exist_ok=True)
os.makedirs("results/runs", exist_ok=True)
print("checkpoints ->", f"{DRIVE_DIR}/outputs")

In [ ]:
# Reproduces data/processed/{train,validation,test}.csv exactly (seeded Phase 1 pipeline)
!python scripts/prepare_data.py
!python scripts/check_tokenizers.py

In [ ]:
import subprocess, shutil, json, pathlib

def train(model, seed, lr=None, tag=None):
    short = model.split("/")[-1]
    name = tag or f"{short}_seed{seed}"
    out = pathlib.Path(f"{DRIVE_DIR}/outputs/{name}")
    cmd = ["python", "scripts/train_transformer.py", "--model", model,
           "--seed", str(seed), "--output-dir", str(out)]
    if lr: cmd += ["--lr", str(lr)]
    print(">>>", " ".join(cmd), flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:]); raise SystemExit(f"training failed: {name}")
    shutil.copy(out / "run_summary.json", f"results/runs/{name}.json")
    print(f"[{name}] best val macro-F1:", json.loads((out/'run_summary.json').read_text())["best_val_macro_f1"])

for model in MODELS:
    for seed in SEEDS:
        train(model, seed)

In [ ]:
# Learning-rate comparison on the best model (by mean validation macro-F1 across seeds).
# Selection uses validation only. The 2e-5 runs already exist; add 5e-5 at seed 42.
import json, glob, statistics, pathlib

by_model = {}
for p in glob.glob("results/runs/*_seed*.json"):
    s = json.loads(pathlib.Path(p).read_text())
    by_model.setdefault(s["model"], []).append(s["best_val_macro_f1"])
means = {m: statistics.mean(v) for m, v in by_model.items()}
best_model = max(means, key=means.get)
print("mean val macro-F1 per model:", {m: round(v, 4) for m, v in means.items()})
print("best model:", best_model)

short = best_model.split("/")[-1]
train(best_model, seed=42, lr=5e-5, tag=f"{short}_seed42_lr5e-5")

In [ ]:
# Test-set evaluation, once per final model (best LR per model chosen on validation)
import subprocess, json, glob, pathlib, statistics

runs = {}
for p in glob.glob("results/runs/*.json"):
    s = json.loads(pathlib.Path(p).read_text())
    runs.setdefault(s["model"], []).append(s)
for model, rs in runs.items():
    best_lr = max(rs, key=lambda r: r["best_val_macro_f1"])["lr"]
    # final run dir = best seed (highest val macro-F1) at the best LR
    final = max((r for r in rs if r["lr"] == best_lr), key=lambda r: r["best_val_macro_f1"])
    print(">>> evaluating", model, "(lr", best_lr, ", seed", final["seed"], ")")
    r = subprocess.run(["python", "scripts/eval_transformer.py",
                        "--model-dir", f"{DRIVE_DIR}/outputs/{model.split('/')[-1]}_seed{final['seed']}",
                        "--name", model.split("/")[-1]], capture_output=True, text=True)
    print(r.stdout[-500:] or r.stderr[-2000:])

In [ ]:
!python scripts/compare_models.py

In [ ]:
# Copy all small artifacts (metrics, charts, reports, error exports) to Drive
!python scripts/export_errors_analysis.py 2>/dev/null || true
!cp results/transformer_*_test_metrics.json results/runs/*.json "{DRIVE_DIR}/" 2>/dev/null; true
!cp results/*comparison*.png results/*confusion*.png "{DRIVE_DIR}/" 2>/dev/null; true
!cp reports/comparison.md "{DRIVE_DIR}/" 2>/dev/null; true
!ls -la "{DRIVE_DIR}/"